In [1]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD DATA
# ==========================

file="../8.Near Miss -Jan to Dec 18.xlsx"

xls=pd.ExcelFile(file)

print("Sheets:")
print(xls.sheet_names)

sheet="Near Miss (1)"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

original_shape=df.shape

print("\nOriginal Shape:",original_shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

df=df.replace(
    [
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "nan",
        "Not Mentioned",
        "not mentioned",
        "NOT MENTIONED"
    ],
    np.nan
)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    if (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    ):

        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

print("\nColumns After Removing 100% Missing:")
print(list(df.columns))

# ==========================
# FIND DESCRIPTION COLUMNS
# ==========================

detail_col=None
event_col=None

for col in df.columns:

    c=col.lower()

    if (
        "details" in c
        and
        "near" in c
    ):

        detail_col=col

    if (
        "description" in c
        and
        "incident" in c
    ):

        event_col=col

# ==========================
# REMOVE ROW ONLY IF BOTH EMPTY
# ==========================

if detail_col and event_col:

    before=len(df)

    keep=(
        (
            df[detail_col]
            .fillna("")
            .astype(str)
            .str.strip()
            !=""
        )
        |
        (
            df[event_col]
            .fillna("")
            .astype(str)
            .str.strip()
            !=""
        )
    )

    df=df[keep]

    print(
        "\nRows Removed (Both Description Columns Empty):",
        before-len(df)
    )

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:",duplicates)

# ==========================
# RESET SERIAL NUMBER
# ==========================

si_col=None

for col in df.columns:

    clean=(
        col.lower()
        .replace("_","")
        .replace(":","")
        .replace(" ","")
    )

    if clean in [
        "slno",
        "sino",
        "serialno",
        "#"
    ]:

        si_col=col
        break

df=df.reset_index(
    drop=True
)

if si_col:

    df[si_col]=range(
        1,
        len(df)+1
    )

    print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")

if len(missing):

    print(missing)

else:

    print("No Missing Values")

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nFinal Columns:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_8_Near_Miss.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Sheets:
['Sheet1', 'Near Miss (1)']

Original Shape: (948, 35)

Removed Empty Columns:
['Corrective_Action', 'Pic', 'CA_Due_Date', 'Status_of_CA', 'Closure_Date_-_CA', 'Remarks_-_CA']

Columns After Removing 100% Missing:
['SI_No', 'Ref_num/Report_date', 'Vessel_Name', 'Vessel_Type', 'Date_and_Time_of_occurrence', 'Period', 'Status', 'Location', 'Port_Name', 'Related_Department', 'Possibility_Of_Recurrence', 'Details_Of_NearMiss', 'Description_of_event_leading_to_the_incident', 'Immediate_Action_Taken', 'Potential_Extent_Of_Damage/Injury', 'Near_Miss_Potential', 'Potential_Damage_category', 'Potential_Damage_subcategory', 'Cause_Analysis', 'Proposed_Corrective_Action', 'Details_Of_Potential_loss_Category', 'Immediate_Action_Initiated', 'Learning_Potential', 'Severity_Of_Incident', 'External_authority_reporting_required', 'Further_Investigation_Required', 'Area_Of_Concern', 'Incident_Communicated_To_Oil_Majors', 'Potential_Incident_To_Be_Shared_With_Industry']

Rows Removed (Both Descri

C:\Users\vinyt\AppData\Local\Temp\ipykernel_23096\3815667948.py:44: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.replace(



Saved: cleaned_8_Near_Miss.xlsx
